# Prepare flight-delay records

Select Atlanta departures, standardize fields, remove incomplete and duplicate records, and export the cleaned CSV consumed by the training notebook. The standalone `src/preprocess.py` also writes the IATA-to-BTS airport mapping used by the API.


## Load the on-time records


In [2]:
# Library Import
import pandas as pd
import json

# Importing Dataset
file_path = r"../data/T_ONTIME_REPORTING.csv"
df = pd.read_csv(file_path)

# Display df
df.head(10)
# Preserve the same IATA aliases and numeric BTS IDs as src/preprocess.py.
destinations = df.loc[df["ORIGIN"].eq("ATL"), ["DEST", "DEST_AIRPORT_ID"]].drop_duplicates()
if not destinations["DEST"].is_unique or not destinations["DEST_AIRPORT_ID"].is_unique:
    raise ValueError("Conflicting airport identifiers for Atlanta departures")
with open("../src/iata_to_bts.json", "w") as alias_file:
    json.dump(dict(zip(destinations["DEST"], destinations["DEST_AIRPORT_ID"].astype(str))), alias_file, indent=2, sort_keys=True)


## Select columns and Atlanta departures


In [3]:
# Column Formatting & Filtering

# | YEAR | MONTH | DAY | DAY_OF_WEEK | ORG_AIRPORT | DEST_AIRPORT | SCHEDULED_DEPARTURE | DEPARTURE_TIME | DEPARTURE_DELAY | SCHEDULED_ARRIVAL | ARRIVAL_TIME | ARRIVAL_DELAY |
# |:--------:|:--------:|:--------:|:--------:|:--------:|:--------:|:--------:|:--------:|:--------:|:--------:|:--------:|:--------:|
# | integer | integer | integer | integer | string | string | integer | integer | integer | integer | integer | integer |


# Select only the required columns and rename them
df = df.rename(columns={
    'DAY_OF_MONTH': 'DAY',
    'ORIGIN_AIRPORT_ID': 'ORG_AIRPORT',
    'DEST_AIRPORT_ID': 'DEST_AIRPORT',
    'CRS_DEP_TIME': 'SCHEDULED_DEPARTURE',
    'DEP_TIME': 'DEPARTURE_TIME',
    'DEP_DELAY': 'DEPARTURE_DELAY',
    'CRS_ARR_TIME': 'SCHEDULED_ARRIVAL',
    'ARR_TIME': 'ARRIVAL_TIME',
    'ARR_DELAY': 'ARRIVAL_DELAY'
})[['YEAR', 'MONTH', 'DAY', 'DAY_OF_WEEK', 'ORG_AIRPORT', 'DEST_AIRPORT', 
     'SCHEDULED_DEPARTURE', 'DEPARTURE_TIME', 'DEPARTURE_DELAY', 
     'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME', 'ARRIVAL_DELAY']]


# Convert all columns except ORG_AIRPORT and DEST_AIRPORT to integer type
# df = df.apply(lambda col: col.astype('Int64') if col.name not in ['ORG_AIRPORT', 'DEST_AIRPORT'] else col)

# Check data types of each column
print(df.dtypes)

display(df)

YEAR                     int64
MONTH                    int64
DAY                      int64
DAY_OF_WEEK              int64
ORG_AIRPORT              int64
DEST_AIRPORT             int64
SCHEDULED_DEPARTURE      int64
DEPARTURE_TIME         float64
DEPARTURE_DELAY        float64
SCHEDULED_ARRIVAL        int64
ARRIVAL_TIME           float64
ARRIVAL_DELAY          float64
dtype: object


,YEAR,MONTH,DAY,DAY_OF_WEEK,ORG_AIRPORT,DEST_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY
0,2025,1,1,3,10135,11057,606,556.0,-10.0,812,736.0,-36.0
1,2025,1,1,3,10135,11057,1219,1215.0,-4.0,1417,1405.0,-12.0
2,2025,1,1,3,10135,11057,1738,1730.0,-8.0,1934,1920.0,-14.0
3,2025,1,1,3,10135,14082,830,820.0,-10.0,1130,1127.0,-3.0
4,2025,1,1,3,10135,14112,1551,1547.0,-4.0,1850,1837.0,-13.0
...,...,...,...,...,...,...,...,...,...,...,...,...
539742,2025,1,31,5,16869,11292,1345,1514.0,89.0,1500,1638.0,98.0
539743,2025,1,31,5,16869,11292,1515,1711.0,116.0,1630,1849.0,139.0
539744,2025,1,31,5,16869,11292,1705,1725.0,20.0,1820,1908.0,48.0
539745,2025,1,31,5,16869,13487,715,706.0,-9.0,927,917.0,-10.0


In [4]:
# Filtering by Origin Airport (ATL)
# Dest Airport ID: 10397

df_atl = df[df['ORG_AIRPORT'] == 10397].copy()

# Display the filtered dataframe
print(df_atl)

df = df_atl

        YEAR  MONTH  DAY  DAY_OF_WEEK  ORG_AIRPORT  DEST_AIRPORT  \
195     2025      1    1            3        10397         10140   
196     2025      1    1            3        10397         10140   
197     2025      1    1            3        10397         10257   
198     2025      1    1            3        10397         10257   
199     2025      1    1            3        10397         10372   
...      ...    ...  ...          ...          ...           ...   
522724  2025      1   31            5        10397         15624   
522725  2025      1   31            5        10397         15624   
522726  2025      1   31            5        10397         15624   
522727  2025      1   31            5        10397         15624   
522728  2025      1   31            5        10397         15624   

        SCHEDULED_DEPARTURE  DEPARTURE_TIME  DEPARTURE_DELAY  \
195                     900           903.0              3.0   
196                    1925          1922.0            

## Handle missing fields and duplicates


In [5]:
# Check for missing values by column
missing_values = df.isnull().sum()

print(missing_values)

YEAR                      0
MONTH                     0
DAY                       0
DAY_OF_WEEK               0
ORG_AIRPORT               0
DEST_AIRPORT              0
SCHEDULED_DEPARTURE       0
DEPARTURE_TIME         1179
DEPARTURE_DELAY        1187
SCHEDULED_ARRIVAL         0
ARRIVAL_TIME           1261
ARRIVAL_DELAY          1301
dtype: int64


In [6]:
# Dropping rows with missing values in specific columns
df.dropna(subset=['DEPARTURE_TIME', 'DEPARTURE_DELAY', 'ARRIVAL_TIME', 'ARRIVAL_DELAY'], inplace=True)

#Ensuring null data is removed
missing_values = df.isnull().sum()

display(missing_values)

YEAR                   0
MONTH                  0
DAY                    0
DAY_OF_WEEK            0
ORG_AIRPORT            0
DEST_AIRPORT           0
SCHEDULED_DEPARTURE    0
DEPARTURE_TIME         0
DEPARTURE_DELAY        0
SCHEDULED_ARRIVAL      0
ARRIVAL_TIME           0
ARRIVAL_DELAY          0
dtype: int64

In [7]:
# Convert ORG_AIRPORT and DEST_AIRPORT to string type
# MLFlow has issues with integer types for these columns - deprecated typing"
df[['ORG_AIRPORT', 'DEST_AIRPORT']] = df[['ORG_AIRPORT', 'DEST_AIRPORT']].astype(str)

# Verify the changes
print(df.dtypes)

YEAR                     int64
MONTH                    int64
DAY                      int64
DAY_OF_WEEK              int64
ORG_AIRPORT             object
DEST_AIRPORT            object
SCHEDULED_DEPARTURE      int64
DEPARTURE_TIME         float64
DEPARTURE_DELAY        float64
SCHEDULED_ARRIVAL        int64
ARRIVAL_TIME           float64
ARRIVAL_DELAY          float64
dtype: object


In [8]:
# Identify duplicate rows
duplicates = df.duplicated()

# Count the number of duplicate rows
num_duplicates = duplicates.sum()
print(f"Number of duplicate rows: {num_duplicates}")

# Display duplicate rows if any
if num_duplicates > 0:
    # Display the duplicate rows before dropping them
    display(df[duplicates])
    
    # Drop duplicate rows
    df.drop_duplicates(inplace=True)
    
    # Recalculate the number of duplicate rows
    new_num_duplicates = df.duplicated().sum()
    
    # Print the number of duplicates removed
    print(f"Number of duplicate rows removed: {num_duplicates - new_num_duplicates}")

Number of duplicate rows: 0


In [9]:
# Dropping spaces before and after each cell
df = df.apply(lambda col: col.map(lambda x: x.strip() if isinstance(x, str) else x))

## Export the cleaned records


In [10]:
# Save the dataframe to a CSV file
df.to_csv('../data/cleaned_data.csv', index=False)